<a href="https://colab.research.google.com/github/leozeze/LRP1_heart/blob/main/02_notebooks/01_download_subset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. 데이터 준비

Heart Cell Atlas v2의 전체 파일(Global, log-norm)을 Colab 임시 공간에 받고,
분석에 필요한 유전자만 추린 **작은 파일**을 구글 드라이브에 저장합니다.

- 원본 파일은 세션이 끝나면 사라집니다 (드라이브에 저장하지 않음)
- 이후 노트북(02, 03)은 여기서 만든 작은 파일만 사용합니다
- 위에서부터 차례로 실행하세요

## 1. 드라이브 연결과 경로 설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/LRP1_heart')      # 드라이브의 프로젝트 폴더
SUBSET_DIR = PROJECT / '01_data' / 'subset'
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = Path('/content/hca_global_lognorm.h5ad')       # 원본을 받을 임시 위치
SUBSET_FILE = SUBSET_DIR / 'lrp1_subset_v1.h5ad'          # 만들 작은 파일
print('작은 파일이 저장될 위치:', SUBSET_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
작은 파일이 저장될 위치: /content/drive/MyDrive/LRP1_heart/01_data/subset/lrp1_subset_v1.h5ad


## 2. 필요한 도구 설치

In [2]:
!pip -q install scanpy "pandas==2.2.3"
import numpy as np, pandas as pd, scipy.sparse as sp
import anndata as ad, scanpy as sc

## 3. 원본 파일 받기

heartcellatlas.org → Data → **Heart Global**의 **H5AD (log-norm)** 아이콘을
우클릭 → 링크 주소 복사 → 아래 따옴표 안에 붙여 넣으세요.

용량이 커서 시간이 걸립니다. 중간에 끊겨도 이 셀을 다시 실행하면 이어서 받습니다.

In [3]:
URL = "https://cellgeni.cog.sanger.ac.uk/heartcellatlas/v2/Global_lognormalised.h5ad"

!df -h /content | tail -1
!wget -c -q --show-progress -O "{RAW_FILE}" "{URL}"
print(f'받은 파일 크기: {RAW_FILE.stat().st_size / 1e9:.1f} GB')

overlay         108G  229M  108G   1% /
/content/hca_global 100%[===================>]   8.13G  34.0MB/s    in 4m 6s   
받은 파일 크기: 8.7 GB


## 4. 파일 구조 확인

파일을 메모리에 올리지 않고 열어서(backed) 어떤 정보가 들어 있는지 봅니다.
여기 출력은 이후 분석에서 열 이름을 맞추는 데 쓰이니 한 번 훑어보세요.

In [4]:
full = sc.read_h5ad(RAW_FILE, backed='r')
print(f'세포/핵 수: {full.n_obs:,}    유전자 수: {full.n_vars:,}')

print('\n[세포별 꼬리표(메타데이터) 열]')
for col in full.obs.columns:
    s = full.obs[col]
    if s.dtype.name in ('category', 'object', 'bool'):
        vals = list(s.astype(str).unique())
        print(f'  {col}: {len(vals)}종류  예) {vals[:6]}')
    else:
        print(f'  {col}: 숫자  ({s.min():.3g} ~ {s.max():.3g})')

print('\n[유전자 이름 예시]', list(full.var_names[:5]))
print('[유전자 정보 열]', list(full.var.columns))
print('[좌표 정보]', list(full.obsm.keys()))

세포/핵 수: 704,296    유전자 수: 32,732

[세포별 꼬리표(메타데이터) 열]
  sangerID: 193종류  예) ['HCAHeart7606896', 'HCAHeart7656534', 'HCAHeart7656535', 'HCAHeart7656536', 'HCAHeart7656537', 'HCAHeart7656538']
  donor: 22종류  예) ['D1', 'D3', 'D4', 'D5', 'D6', 'D7']
  donor_type: 2종류  예) ['DBD', 'DCD']
  region: 8종류  예) ['AX', 'LV', 'RV', 'LA', 'SP', 'RA']
  age: 9종류  예) ['50-55', '55-60', '70-75', '65-70', '60-65', '40-45']
  gender: 2종류  예) ['Female', 'Male']
  facility: 2종류  예) ['Sanger', 'Harvard']
  cell_or_nuclei: 2종류  예) ['Cell', 'Nuclei']
  modality: 3종류  예) ['scRNA', 'snRNA', 'Multiome-RNA']
  kit_10x: 3종류  예) ['3prime-v2', '3prime-v3', 'Multiome-v1']
  flushed: 2종류  예) ['no', 'yes']
  cell_type: 12종류  예) ['Endothelial cell', 'Mural cell', 'Myeloid', 'Fibroblast', 'Lymphoid', 'Neural cell']
  cell_state: 76종류  예) ['EC5_art', 'SMC2_art', 'PC2_atria', 'EC6_ven', 'SMC1_basic', 'EC2_cap']
  n_genes: 숫자  (200 ~ 1.6e+04)
  n_genes_by_counts: 숫자  (200 ~ 1.6e+04)
  total_counts: 숫자  (500 ~ 4.38e+05)
  tota

## 5. 추릴 유전자 목록

원본을 다시 받는 것이 번거로우므로, 나중에 필요할 수 있는 유전자를 넉넉히 넣어 둡니다.
추가하고 싶은 유전자가 있으면 지금 목록에 넣으세요.

In [5]:
GENES = {
    # 분석 대상
    'target':      ['LRP1'],
    'sheddase':    ['ADAM10', 'ADAM12', 'ADAM17', 'MMP14', 'MMP15', 'MMP16', 'MMP17', 'BACE1'],
    # 세포 유형 표지: Litviňuková et al. 2020, Fig. 1c와 본문
    'marker':      ['MYH7', 'MYL2', 'FHL2', 'NPPA', 'MYL7', 'MYL4', 'TTN',
                    'DCN', 'GSN', 'PDGFRA', 'VWF', 'PECAM1', 'CDH5',
                    'RGS5', 'ABCC9', 'KCNJ8', 'MYH11', 'TAGLN', 'ACTA2',
                    'CD14', 'C1QA', 'CD68', 'CD8A', 'IL7R', 'CD40LG',
                    'GPAM', 'FASN', 'LEP', 'ADIPOQ',
                    'PLP1', 'NRXN1', 'NRXN3', 'MSLN', 'WT1', 'BNC1'],
    # 섬유모세포 집단 표지(Fig. 3g)와 vCM3 스트레스 표지(본문)
    'state':       ['SCN7A', 'BMPER', 'ACSM1', 'CFH', 'ID4', 'KCNT2',
                    'PTX3', 'OSMR', 'IL6ST', 'POSTN', 'TNC', 'FAP',
                    'FBLN2', 'PCOLCE2', 'LINC01133', 'CD36', 'EGFLAM',
                    'ANKRD1', 'FHL1', 'XIRP1', 'XIRP2', 'CNN1', 'MYH9', 'NEXN'],
}
wanted = [g for group in GENES.values() for g in group]
print(f'요청한 유전자: {len(wanted)}개')

요청한 유전자: 68개


## 6. 필요한 유전자만 추려 작은 파일 만들기

원본을 5만 세포씩 끊어 읽으면서 위 유전자의 값만 남깁니다.
세포별 꼬리표와 UMAP 좌표는 전부 그대로 가져갑니다.

In [6]:
# 유전자 이름 찾기 (파일에 따라 이름이 다른 열에 들어 있을 수 있음)
symbols = pd.Index(full.var_names.astype(str))
if 'LRP1' not in symbols:
    for col in full.var.columns:
        if 'LRP1' in set(full.var[col].astype(str)):
            symbols = pd.Index(full.var[col].astype(str))
            print(f"유전자 이름으로 '{col}' 열을 사용합니다")
            break

found = [g for g in wanted if g in symbols]
missing = [g for g in wanted if g not in symbols]
print(f'찾은 유전자 {len(found)}개')
if missing:
    print('파일에 없는 유전자:', missing)

pos = {g: int(np.flatnonzero(symbols == g)[0]) for g in found}
order = sorted(found, key=pos.get)
idx = [pos[g] for g in order]

# 끊어 읽기
parts, step = [], 50_000
for start in range(0, full.n_obs, step):
    block = full.X[start:start + step]
    parts.append(sp.csr_matrix(block)[:, idx])
    print(f'  읽는 중: {min(start + step, full.n_obs):,} / {full.n_obs:,}', end='\r')

group_of = {g: grp for grp, gs in GENES.items() for g in gs}
subset = ad.AnnData(
    X=sp.vstack(parts).tocsr().astype(np.float32),
    obs=full.obs.copy(),
    var=pd.DataFrame({'group': [group_of[g] for g in order]}, index=order),
)
for key in full.obsm.keys():                      # UMAP 좌표 등
    subset.obsm[key] = np.asarray(full.obsm[key])
subset.uns['source'] = 'Heart Cell Atlas v2, Global, log-normalised'

subset.write_h5ad(SUBSET_FILE, compression='gzip')
print(f'\n저장 완료: {SUBSET_FILE}  ({SUBSET_FILE.stat().st_size / 1e6:.0f} MB)')

찾은 유전자 68개

저장 완료: /content/drive/MyDrive/LRP1_heart/01_data/subset/lrp1_subset_v1.h5ad  (77 MB)


## 7. 저장된 파일 점검

작은 파일을 다시 열어 제대로 만들어졌는지 확인합니다.
표지 유전자가 알려진 세포 유형에서 가장 높게 나오면 정상입니다
(예: DCN은 섬유모세포, TTN은 심근세포, PTPRC는 면역세포).

In [7]:
check = sc.read_h5ad(SUBSET_FILE)
print(check)

markers = [g for g in ['DCN', 'TTN', 'VWF', 'CD68', 'ADIPOQ'] if g in check.var_names]
X = pd.DataFrame(check[:, markers].X.toarray() > 0, columns=markers)
pct = X.groupby(check.obs['cell_type'].astype(str).values).mean().mul(100).round(1)
print('\n[세포 유형별 표지 유전자 검출 비율 (%)]')
pct

AnnData object with n_obs × n_vars = 704296 × 68
    obs: 'sangerID', 'donor', 'donor_type', 'region', 'age', 'gender', 'facility', 'cell_or_nuclei', 'modality', 'kit_10x', 'flushed', 'cell_type', 'cell_state', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'scrublet_score'
    var: 'group'
    uns: 'source'
    obsm: 'X_umap'

[세포 유형별 표지 유전자 검출 비율 (%)]


,DCN,TTN,VWF,CD68,ADIPOQ
Adipocyte,28.3,20.1,7.0,1.9,66.7
Atrial Cardiomyocyte,2.4,98.5,2.7,0.2,0.1
Endothelial cell,2.0,5.4,82.5,0.6,0.0
Fibroblast,88.7,12.2,1.9,3.3,0.4
Lymphatic Endothelial cell,8.1,23.2,24.9,4.4,0.6
Lymphoid,0.9,3.3,1.1,1.1,0.0
Mast cell,1.6,11.3,1.5,0.3,0.3
Mesothelial cell,22.4,14.7,7.4,2.7,2.3
Mural cell,15.9,5.0,2.2,0.4,0.0
Myeloid,3.2,29.3,2.2,36.8,0.1
